# Method 1 鈥?DFJ formulation (Gurobi)

The **Dantzig鈥揊ulkerson鈥揓ohnson** model uses one binary variable per directed arc and
eliminates every possible subtour explicitly.

$$
\begin{aligned}
\min\quad & \sum_{i}\sum_{j\ne i} d_{ij}x_{ij} \\
\text{s.t.}\quad & \sum_{j\ne i}x_{ij}=1 && \forall i \\
& \sum_{j\ne i}x_{ji}=1 && \forall i \\
& \sum_{i\in S}\sum_{j\in S, j\ne i}x_{ij}\le |S|-1 && \forall S\subset V\setminus\{0\},\ 2\le|S|\le n-1 \\
& x_{ij}\in\{0,1\}
\end{aligned}
$$

For 14 cities there are $2^{13}-14=8{,}178$ proper subsets of the non-depot cities of size at least two, so the model
has **8,178 subset cuts**. That is small enough for Gurobi, and the formulation has a very
tight LP relaxation.

> Gurobi is optional. If `gurobipy` or a licence is missing, the notebook falls back to the
> exact Held鈥揔arp dynamic program so the route is still reproducible.

## 1. Load the data and build the distance matrix

In [ ]:
from tsp_solver import (
    DEFAULT_DATA, load_cities, distance_matrix,
    solve_dfj, held_karp, GurobiUnavailable,
    plot_route, route_names,
)

cities = load_cities(DEFAULT_DATA)
distances = distance_matrix(cities)
cities

## 2. Solve with the DFJ model

In [ ]:
try:
    result = solve_dfj(distances, time_limit=120, output_flag=False)
except GurobiUnavailable as error:
    print(f"[fallback] {error}")
    result = held_karp(distances)

print(result.method)
print(f"cost: {result.cost:.2f} km")
print("tour:", " -> ".join(route_names(result.tour, cities)))

## 3. Plot the optimal route

In [ ]:
plot_route(cities, result.tour, result.cost, "results/dfj_route.png", "DFJ route")
from IPython.display import Image
Image(filename="results/dfj_route.png")

## Why it works

- Degree constraints force exactly one arc into and out of every city.
- Without the subset constraints, the solver can return several disconnected cycles.
- The subset constraints require at least one arc to leave every proper subset of
  non-depot cities, which rules out every subtour.
- The model is exponential in the number of cities; it is ideal for small, exact instances
  such as this 14-city teaching example.